# World Commodities (Yahoo Finance)

In [1]:
from typing import cast

import pandas as pd
import yfinance as yf
import mgplot as mg

In [2]:
CHART_DIR = "./CHARTS/Commodities/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False

In [3]:
# --- Fetch data ---
tickers = {"CL=F": "WTI", "BZ=F": "Brent"}
frames = {}
for ticker, label in tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        frames[label] = df["Close"].squeeze()

prices = pd.DataFrame(frames).dropna()
prices.index = cast(pd.DatetimeIndex, prices.index).to_period("D")

In [4]:
# --- Data summary ---
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
for col in prices.columns:
    print(f"  {col:5s}  min={prices[col].min():.2f}  max={prices[col].max():.2f}")

Date range: 2025-12-01 to 2026-03-19
  WTI    min=55.27  max=98.71
  Brent  min=58.92  max=107.38


In [5]:
# --- Chart 1: Price Series ---
mg.line_plot_finalise(
    prices,
    title="Crude Oil Prices: WTI vs Brent",
    ylabel="USD per barrel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter="NYMEX (WTI) and ICE (Brent) daily closing prices",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

print(f"\nChart saved to {CHART_DIR}")


Chart saved to ./CHARTS/Commodities/


In [6]:
# --- Fetch gold data (last 2 years) ---
gold_df = yf.download("GC=F", start="2024-03-20", auto_adjust=True, progress=False)
if gold_df is not None:
    gold = gold_df["Close"].squeeze().dropna()
    gold.index = cast(pd.DatetimeIndex, gold.index).to_period("D")

print(f"Gold date range: {gold.index[0]} to {gold.index[-1]}")
print(f"  Gold  min={gold.min():.2f}  max={gold.max():.2f}")

Gold date range: 2024-03-20 to 2026-03-19
  Gold  min=2157.90  max=5318.40


In [7]:
# --- Chart 2: Gold Price ---
mg.line_plot_finalise(
    gold,
    title="Gold Price",
    ylabel="USD per troy ounce",
    xlabel=None,
    lfooter="COMEX daily closing price",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [8]:
# --- Fetch copper data (last 2 years) ---
copper_df = yf.download("HG=F", start="2024-03-20", auto_adjust=True, progress=False)
if copper_df is not None:
    copper = copper_df["Close"].squeeze().dropna()
    copper.index = cast(pd.DatetimeIndex, copper.index).to_period("D")

print(f"Copper date range: {copper.index[0]} to {copper.index[-1]}")
print(f"  Copper  min={copper.min():.4f}  max={copper.max():.4f}")

Copper date range: 2024-03-20 to 2026-03-19
  Copper  min=3.9375  max=6.1755


In [9]:
# --- Chart 3: Copper Price ---
mg.line_plot_finalise(
    copper,
    title="Copper Price",
    ylabel="USD per pound",
    xlabel=None,
    lfooter="COMEX daily closing price",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [10]:
# --- Fetch natural gas data ---
gas_tickers = {
    "NG=F": "Henry Hub (US)",
    "TTF=F": "TTF (Europe)",
    "JKM=F": "JKM (Asia)",
}
gas_frames = {}
for ticker, label in gas_tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        gas_frames[label] = s

# Fetch EUR/USD rate to convert TTF from EUR/MWh to USD/MMBtu
fx_df = yf.download("EURUSD=X", start="2025-12-01", auto_adjust=True, progress=False)
if fx_df is not None:
    eurusd = fx_df["Close"].squeeze().dropna()
    eurusd.index = cast(pd.DatetimeIndex, eurusd.index).to_period("D")

gas_prices = pd.DataFrame(gas_frames).dropna()

# Convert TTF: EUR/MWh -> USD/MMBtu (1 MWh = 3.412 MMBtu)
MMBTU_PER_MWH = 3.412
ttf_aligned = eurusd.reindex(gas_prices.index, method="ffill")
gas_prices["TTF (Europe)"] = gas_prices["TTF (Europe)"] * ttf_aligned / MMBTU_PER_MWH

print(f"Natural Gas date range: {gas_prices.index[0]} to {gas_prices.index[-1]}")
for col in gas_prices.columns:
    print(f"  {col:20s}  min={gas_prices[col].min():.3f}  max={gas_prices[col].max():.3f}")

Natural Gas date range: 2025-12-01 to 2026-03-19
  Henry Hub (US)        min=2.827  max=7.460
  TTF (Europe)          min=9.066  max=20.642
  JKM (Asia)            min=9.455  max=22.350


In [11]:
# --- Chart 4: Natural Gas Benchmarks ---
mg.line_plot_finalise(
    gas_prices,
    title="Natural Gas Benchmarks",
    ylabel="USD per MMBtu",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter="TTF converted from EUR/MWh using daily EUR/USD rate",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [12]:
# --- Fetch grains data (last 2 years) ---
grain_tickers = {"ZW=F": "Wheat", "ZC=F": "Corn"}
grain_frames = {}
for ticker, label in grain_tickers.items():
    df = yf.download(ticker, start="2024-03-20", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        grain_frames[label] = s

grains = pd.DataFrame(grain_frames).dropna()

print(f"Grains date range: {grains.index[0]} to {grains.index[-1]}")
for col in grains.columns:
    print(f"  {col:6s}  min={grains[col].min():.2f}  max={grains[col].max():.2f}")

Grains date range: 2024-03-20 to 2026-03-19
  Wheat   min=495.00  max=700.25
  Corn    min=362.00  max=502.00


In [13]:
# --- Chart 5: Grain Prices ---
mg.line_plot_finalise(
    grains,
    title="Grain Prices: Wheat and Corn",
    ylabel="USD cents per bushel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    lfooter="CBOT daily closing prices",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [14]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-03-20 14:21:39

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.11.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.21
pandas  : 3.0.1
typing  : 3.10.0.0
yfinance: 1.2.0

Watermark: 2.6.0



In [15]:
print("Finished")

Finished
